# Scoring Harnesses: CIDEr and COCO mAP
**DS 677 · Florence-2 vs. Specialist Models Under Motion Blur** · Vivek Shah and Victor Figueredo

Every result in this project is a score: **CIDEr** for captioning and **mAP** for object detection. This notebook tests the shared scoring code in `src/evaluate.py` before any real experiment depends on it, then uses it for a first real result: DETR on 50 images, scored two ways.

### What `src/evaluate.py` provides
| Function | What it does |
|---|---|
| `score_captions(predictions, references)` | CIDEr and BLEU-4 against the human captions, using the official COCO tokenizer (×100, so Florence-2's reported score reads as 140) |
| `to_coco_box(box)` | Converts a box from `[x1, y1, x2, y2]` (model output) to `[x, y, width, height]` (COCO format) |
| `score_detections(detections, coco, image_ids)` | Official COCO evaluation: mAP, mAP@0.5, and AP for small, medium and large objects |
| `save_result(row)` | Adds one row to the results CSV using the agreed columns in `src/config.py` |

`src/evaluate.py` is a normal Python file in the repo. This notebook only **imports** it, so there is a single copy of the code.

### What this notebook does
| What happens |
|---|
| Install packages and get the team's latest code |
| Download the detection data and the Karpathy caption file (~1 GB) |
| Test the caption scorer with human captions |
| Test the detection scorer with the ground-truth boxes |
| Check DETR's class numbers match COCO's |
| Run DETR on 50 images and score it two ways |
| Save results in the shared format |


## Step 1 · Install packages and get the team's code
- Installs the pinned packages, plus **`pycocoevalcap`**, the official COCO caption-scoring package.
- Checks **Java** is installed: the official caption tokenizer is a Java program.
- Clones the latest repo, which includes `src/evaluate.py`, and lets Python import from `src/`.

**Result:** Java 21 is available and the latest commit is shown. The pip warnings about `gradio` and `diffusers` are expected and harmless.

In [1]:
# Install pinned package versions and pycocoevalcap for caption scoring
!pip install -q transformers==4.49.0 timm einops pycocotools==2.0.11 pycocoevalcap==1.2

# Caption Scoring needs Java
!java -version

# Download a fresh copy of the team's repo and move into it
!rm -rf /content/repo
!git clone -q https://github.com/NJIT-DS677-Deep-Learning/florence2-vs-specialists-motion-blur.git /content/repo
%cd /content/repo
# Show the latest commit
!git log -1 --oneline

# Let Python import modules from src/
import sys
sys.path.insert(0, '/content/repo')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
[0.014s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.014s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-2

## Step 2 · Download the data this notebook needs
Only about 1 GB is needed here, compared with about 7 GB for the full project:
- **val2017** images and **annotations** for the detection tests.
- **`dataset_coco.json`** for the captioning tests. These only need the human captions, not the images, so `val2014` is skipped.

**Result:** `/content/data` contains `annotations`, `dataset_coco.json` and `val2017`.

In [2]:
# Create the data folder outside the repo
!mkdir -p /content/data
%cd /content/data

# Detection images, detection ground truth, and the Karpathy caption file
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip

# Unpack, put dataset_coco.json in place, and delete the zips
!unzip -q -o val2017.zip
!unzip -q -o annotations_trainval2017.zip
!unzip -q -o caption_datasets.zip
!mv $(find /content/data -name dataset_coco.json | head -1) /content/data/ 2>/dev/null
!rm -f *.zip

# Check what was downloaded, then go back to the repo
!ls /content/data
%cd /content/repo

/content/data
val2017.zip         100%[===================>] 777.80M  15.6MB/s    in 1m 47s  
annotations_trainva 100%[===================>] 241.19M  49.7MB/s    in 5.5s    
caption_datasets.zi 100%[===================>]  35.04M  3.33MB/s    in 9.5s    
annotations	   dataset_flickr30k.json  val2017
dataset_coco.json  dataset_flickr8k.json
/content/repo


## Step 3 · Test the caption scorer with human captions
Checks that `score_captions()` can tell good captions from bad ones, without running any model:

- **Test 1 (human vs. human):** each image's first human caption is scored as if a model wrote it, against the image's other human captions. This should give a **human-level score, roughly 70–100 CIDEr**.
- **Test 2 (shuffled):** every image is given a caption from a *different* image. A working scorer must give this **close to 0**.

Models will later score *higher* than humans (Florence-2 reports 140 CIDEr). That is a known property of CIDEr: models produce the most typical wording, which matches many references at once, while each person phrases things their own way.

**Result: both tests passed.**

| Test | Expected | CIDEr | BLEU-4 |
|---|---|---|---|
| Human vs. human | roughly 70–100 | **89.3** | 19.9 |
| Shuffled captions | close to 0 | **3.1** | 1.9 |

Human captions score 89.3, close to the human score of about 85 on the official COCO captioning leaderboard, while mismatched captions score almost nothing. The scorer clearly tells correct captions from wrong ones.

In [3]:
import random
from src.data import load_karpathy_test, load_val2017, open_image
from src.evaluate import score_captions, score_detections, to_coco_box, save_result

# Load the 5,000 Karpathy test images and their human captions
captions = load_karpathy_test()

# Test 1: score each image's first human caption against its other human captions
predictions = {x["image_id"]: x["references"][0] for x in captions}
references = {x["image_id"]: x["references"][1:] for x in captions}
print("Human vs. Human:", score_captions(predictions, references))

# Test 2: give every image a caption from a DIFFERENT image (should score near zero)
random.seed(0)
ids = list(predictions)
shuffled = dict(zip(ids, random.sample(list(predictions.values()), len(ids))))
print("Shuffled Captions:", score_captions(shuffled, references))

Human vs. Human: {'CIDEr': 89.28043862909097, 'BLEU-4': 19.913156913595675}
Shuffled Captions: {'CIDEr': 3.1175315232311847, 'BLEU-4': 1.9004578176140356}


## Step 4 · Test the detection scorer with the ground-truth boxes
The detection version of the same idea, run on all 5,000 val2017 images:

- **Test 3:** the true boxes are given as "detections". This must score a **perfect 100 mAP**.
- **Test 4:** only every other true box is kept. This should score **about 50**.

The block of *Average Precision* lines is pycocotools' standard printout; the dictionary at the end is what `score_detections()` returns.

**Result: both tests passed.**

| Test | Expected | Got |
|---|---|---|
| Perfect detections | 100 mAP | 100.0 mAP (100 for small, medium and large objects) |
| Half the detections | about 50 mAP | 50.2 mAP |

In [4]:
# Load val2017: the image list and the ground-truth boxes
detection, coco = load_val2017()
image_ids = [x["image_id"] for x in detection]

# Test 3: use the true boxes as "detections" (should score a perfect 100)
true_boxes = coco.loadAnns(coco.getAnnIds(imgIds=image_ids, iscrowd=False))
perfect = [{"image_id": a["image_id"], "category_id": a["category_id"], "bbox": a["bbox"], "score": 1.0} for a in true_boxes]
print("Perfect detections:", score_detections(perfect, coco, image_ids))

# Test 4: keep only every other true box (should score about 50)
print("Half the detections:", score_detections(perfect[::2], coco, image_ids))

loading annotations into memory...
Done (t=0.90s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.16s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.65s).
Accumulating evaluation results...
DONE (t=4.39s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.563
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.976
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

## Step 5 · Check DETR's class numbers match COCO's
DETR outputs a class *number* for each object, and the scorer needs COCO's category IDs. This compares every COCO category name with DETR's name for the same number.

**Result:** `Categories that don't match: []`. DETR uses exactly COCO's numbering, so its class numbers can be passed to the scorer directly.

In [5]:
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection

# Use the GPU if one is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Load DETR and its image processor
detr_proc = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device)

# Compare every COCO category name with DETR's name for the same number
mismatches = [(c["id"], c["name"], detr.config.id2label.get(c["id"])) for c in coco.cats.values() if detr.config.id2label.get(c["id"]) != c["name"]]
print("Categories that don't match:", mismatches)

Running on: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Categories that don't match: []


## Step 6 · DETR on 50 images, scored two ways
The first real model score. Each DETR prediction is stored under two protocols:

- **Official:** every prediction with DETR's own confidence score. This is how DETR's published 42.0 AP was measured.
- **Matched:** only predictions at least 50% confident, all given the same score of 1.0. Florence-2 outputs no confidence scores, so this is the fair way to compare DETR with it.

**Result:**

| Protocol | mAP | mAP@0.5 | AP small | AP medium | AP large |
|---|---|---|---|---|---|
| Official | 46.8 | 65.7 | 24.8 | 55.2 | 72.1 |
| Matched | 39.7 | 57.4 | 17.6 | 45.5 | 65.8 |

- The official score is in the expected range. It is above the published 42.0 because 50 images is a small, noisy sample.
- The matched protocol is **7 points lower**. Without confidence scores, mAP cannot rank the most certain detections first. This is why the project reports both numbers, and why Florence-2 should be compared against DETR's *matched* score.
- Small objects score far lower than large ones (24.8 vs. 72.1), matching DETR's known weakness on small objects. Blur is likely to hurt small objects the most, which is worth watching in the analysis.

In [6]:
# Choose a subset of 50 val2017 images
N = 50
subset = detection[:N]
subset_ids = [x["image_id"] for x in subset]

official, matched = [], []
for x in subset:
  img = open_image(x["path"])

  # Run DETR and keep every prediction with its confidence score
  inputs = detr_proc(images=img, return_tensors="pt").to(device)
  with torch.no_grad():
    output = detr(**inputs)
  result = detr_proc.post_process_object_detection(output, threshold=0.0, target_sizes=[img.size[::-1]])[0]

  for score, label, box in zip(result["scores"], result["labels"], result["boxes"]):
    det = {"image_id": x["image_id"], "category_id": int(label), "bbox": to_coco_box(box.tolist()), "score": float(score)}
    # Official protocol: every prediction, with DETR's own confidence
    official.append(det)
    # Matched protocol: only confident predictions, all given the same score (like Florence-2)
    if score >= 0.5:
      matched.append({**det, "score": 1.0})

# Score both protocols on the same 50 images
official_scores = score_detections(official, coco, subset_ids)
matched_scores = score_detections(matched, coco, subset_ids)
print("Official Protocol:", official_scores)
print("Matched Protocol: ", matched_scores)

Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.76s).
Accumulating evaluation results...
DONE (t=0.64s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.468
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.657
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.495
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.248
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.552
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.721
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.364
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.566
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.583
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

## Step 7 · Save results in the shared format
Checks that `save_result()` writes rows with the columns agreed in `src/config.py`. It deliberately writes to a **test file outside the repo**, so smoke-test numbers never mix with the real results. The file is deleted first, so rerunning the cell does not add duplicate rows.

**Result:** a header row matching `RESULTS_COLUMNS`, followed by one row per protocol (46.77 official, 39.74 matched). The empty `seed` column is expected, since no blur was applied.

In [7]:
import datetime
# Today's date for the results row
today = datetime.date.today().isoformat()

# Start the TEST results file fresh, so reruns do not add duplicate rows
!rm -f /content/test_results.csv

# Write one row per protocol to a TEST results file (not the real one)
for protocol, scores in [("official", official_scores), ("matched", matched_scores)]:
  save_result({"model": "detr-resnet-50", "task": "detection", "blur_type": "none", "blur_px": 0, "metric": "mAP", "score": round(scores["mAP"], 2), "n_images": N, "date": today, "notes": f"{protocol} protocol, 50-image smoke test"}, path="/content/test_results.csv")

!cat /content/test_results.csv

model,task,blur_type,blur_px,seed,metric,score,n_images,date,notes
detr-resnet-50,detection,none,0,,mAP,46.77,50,2026-11-13,"official protocol, 50-image smoke test"
detr-resnet-50,detection,none,0,,mAP,39.74,50,2026-11-13,"matched protocol, 50-image smoke test"


## Summary
- **Caption Scorer:** `score_captions()` uses the official COCO tokenizer and CIDEr. Human captions score 89.3 CIDEr and mismatched captions 3.1.
- **Detection Scorer:** `score_detections()` gives exactly 100 mAP for perfect detections and about 50 for half of them, so box format and scoring are correct. DETR's class numbers match COCO's.
- **Two DETR protocols:** on 50 images, DETR scores 46.8 mAP officially and 39.7 with the matched, Florence-2-style protocol.
- **Results format:** `save_result()` writes rows with the agreed columns.